# 01 — EDA y Limpieza del Dataset LSGoodreads

**Editorial ficticia**: Historia & Biografía
**Dataset**: `src/data/Best_History_Books/` — JSONs individuales scrapeados de Goodreads

Este notebook carga todos los JSONs, documenta la estructura del dataset, detecta y corrige problemas de calidad y exporta el DataFrame limpio a `src/data/books_clean.csv`.

In [ ]:
import json
import glob
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 4)})
sns.set_theme(style="whitegrid", palette="muted")

RANDOM_SEED = 42
DATA_DIR = os.path.join("..", "src", "data", "Best_History_Books")
OUT_CSV   = os.path.join("..", "src", "data", "books_clean.csv")

## 1. Carga de datos

In [ ]:
paths   = glob.glob(os.path.join(DATA_DIR, "*.json"))
records = []
for p in paths:
    with open(p, encoding="utf-8") as f:
        records.append(json.load(f))

df_raw = pd.DataFrame(records)
print(f"Ficheros cargados : {len(paths):,}")
print(f"Filas x columnas  : {df_raw.shape}")
df_raw.head(3)

## 2. Examen preliminar

### Esquema del dataset

| Campo | Tipo raw | Descripción |
|-------|----------|-------------|
| `book_id` | str | Identificador único de Goodreads (`id-slug`) |
| `book_title` | str | Título del libro (ASCII via unidecode) |
| `author` | str | Autor principal |
| `book_description` | str | Sinopsis larga |
| `genres` | list[str] | Géneros asignados por la comunidad (puede incluir `"...more"`) |
| `average_rating` | str→float | Valoración media 0–5 |
| `num_ratings` | str→int | Número de valoraciones |
| `num_pages` | str→int | Páginas (puede ser formato no numérico: "Paperback", "Nook"…) |
| `year_published` | str→int | Año de publicación original |
| `scrape_date` | str | Timestamp del scraping |

In [ ]:
print("=== Tipos de datos ===")
print(df_raw.dtypes)
print()
print("=== Estadísticas generales (antes de conversión de tipos) ===")
df_raw.describe(include="all").T

## 3. Detección de problemas de calidad

In [ ]:
# ── 3.1 Nulos y cadenas vacías ──────────────────────────────
empty_counts = (df_raw == "").sum()
null_counts  = df_raw.isnull().sum()
print("Campos vacíos (""):")
print(empty_counts[empty_counts > 0])
print()
print("Nulos reales (NaN):")
print(null_counts[null_counts > 0])

In [ ]:
# ── 3.2 Campo genres ────────────────────────────────────────
has_more    = df_raw["genres"].apply(lambda g: "...more" in g if isinstance(g, list) else False)
empty_genre = df_raw["genres"].apply(lambda g: len(g) == 0 if isinstance(g, list) else True)

print(f"Libros con '...more' en genres : {has_more.sum():,}  ({has_more.mean()*100:.1f}%)")
print(f"Libros con genres vacío        : {empty_genre.sum():,}  ({empty_genre.mean()*100:.1f}%)")

In [ ]:
# ── 3.3 num_pages no numérico ───────────────────────────────
non_numeric_pages = df_raw[pd.to_numeric(df_raw["num_pages"], errors="coerce").isna()]
print(f"Filas con num_pages no numérico: {len(non_numeric_pages)}")
print(non_numeric_pages[["book_title", "num_pages"]].head(10).to_string(index=False))

In [ ]:
# ── 3.4 year_published ──────────────────────────────────────
years_num = pd.to_numeric(df_raw["year_published"], errors="coerce")
print(f"year_published vacío / no numérico : {years_num.isna().sum()}")
print(f"Años < 1000                        : {(years_num < 1000).sum()}")
print(f"Años > 2026                        : {(years_num > 2026).sum()}")
print(f"Rango válido                       : {years_num[years_num.between(1000, 2026)].min()} – {years_num[years_num.between(1000, 2026)].max()}")

In [ ]:
# ── 3.5 Duplicados ──────────────────────────────────────────
dups = df_raw.duplicated(subset="book_id", keep=False)
print(f"Filas duplicadas por book_id : {dups.sum()}")
if dups.sum() > 0:
    print(df_raw[dups][["book_id", "book_title"]].head())

## 4. Limpieza

In [ ]:
df = df_raw.copy()

# ── 4.1 Eliminar duplicados ──────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset="book_id", keep="first")
print(f"Duplicados eliminados : {before - len(df)}")

# ── 4.2 Limpiar genres ──────────────────────────────────────
df["genres"] = df["genres"].apply(
    lambda g: [x for x in g if x != "...more"] if isinstance(g, list) else []
)

# ── 4.3 Conversión de tipos ─────────────────────────────────
df["average_rating"] = pd.to_numeric(df["average_rating"], errors="coerce")
df["num_ratings"]    = pd.to_numeric(df["num_ratings"],    errors="coerce").astype("Int64")
df["num_pages"]      = pd.to_numeric(df["num_pages"],      errors="coerce")
df["year_published"] = pd.to_numeric(df["year_published"], errors="coerce")

# ── 4.4 Años imposibles → NaN ────────────────────────────────
df.loc[~df["year_published"].between(1000, 2026), "year_published"] = pd.NA

# ── 4.5 Flag: libro con valoraciones ────────────────────────
df["has_ratings"] = df["num_ratings"].fillna(0) > 0

print("Tipos tras limpieza:")
print(df.dtypes)

In [ ]:
# ── 4.6 Resumen post-limpieza ────────────────────────────────
total      = len(df)
with_rating = df["has_ratings"].sum()
no_pages   = df["num_pages"].isna().sum()
no_year    = df["year_published"].isna().sum()
no_genres  = df["genres"].apply(len).eq(0).sum()

summary = pd.DataFrame({
    "Métrica": [
        "Libros totales",
        "Con valoraciones (num_ratings > 0)",
        "Sin num_pages",
        "Sin year_published",
        "Sin géneros",
    ],
    "Valor": [total, int(with_rating), int(no_pages), int(no_year), int(no_genres)],
    "%": [100, with_rating/total*100, no_pages/total*100, no_year/total*100, no_genres/total*100],
})
summary["%"] = summary["%"].map("{:.1f}%".format)
print(summary.to_string(index=False))

## 5. Visualizaciones exploratorias

In [ ]:
# ── Plot 1: Distribución de average_rating ───────────────────
fig, ax = plt.subplots()
ax.hist(df["average_rating"].dropna(), bins=40, color="steelblue", edgecolor="white")
ax.axvline(df["average_rating"].mean(), color="crimson", linewidth=1.5, linestyle="--",
           label=f'Media = {df["average_rating"].mean():.2f}')
ax.set(title="Distribución de average_rating", xlabel="Rating medio", ylabel="Número de libros")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: num_ratings (escala log) ─────────────────────────
fig, ax = plt.subplots()
vals = df.loc[df["has_ratings"], "num_ratings"].dropna().astype(float)
ax.hist(np.log10(vals + 1), bins=50, color="teal", edgecolor="white")
ax.set(title="Distribución de num_ratings (log₁₀)", xlabel="log₁₀(num_ratings)", ylabel="Número de libros")
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: num_pages (boxplot + strip) ──────────────────────
pages_clean = df["num_pages"].dropna()
pages_clean = pages_clean[(pages_clean > 0) & (pages_clean < 2000)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(pages_clean, vert=True, patch_artist=True,
                boxprops=dict(facecolor="lightyellow"))
axes[0].set(title="Boxplot — num_pages", ylabel="Páginas")
axes[1].hist(pages_clean, bins=60, color="goldenrod", edgecolor="white")
axes[1].set(title="Histograma — num_pages", xlabel="Páginas", ylabel="Frecuencia")
plt.tight_layout()
plt.show()
print(pages_clean.describe().to_string())

In [ ]:
# ── Plot 4: Top-20 géneros más frecuentes ────────────────────
from collections import Counter

genre_counts = Counter(g for genres in df["genres"] for g in genres)
top_genres   = pd.Series(genre_counts).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 5))
top_genres.sort_values().plot(kind="barh", ax=ax, color="cornflowerblue", edgecolor="white")
ax.set(title="Top 20 géneros más frecuentes", xlabel="Número de libros", ylabel="")
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 5: Publicaciones por año ────────────────────────────
year_counts = (
    df.dropna(subset=["year_published"])
      .query("year_published >= 1900")
      ["year_published"]
      .astype(int)
      .value_counts()
      .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(year_counts.index, year_counts.values, color="mediumseagreen", width=0.8)
ax.set(title="Libros publicados por año (desde 1900)", xlabel="Año", ylabel="Número de libros")
ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Exportación

In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f"Dataset limpio guardado en: {OUT_CSV}")
print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
df.head(3)